# OpenJev: zero-shot inference

Chạy OpenJev Flash 9B **gốc, BF16**, chưa fine-tune trên dataset của bạn. Không tạo/load LoRA, không cần train.tsv, không nạp mẫu tổng hợp. Dùng cùng câu hỏi, logits A/B, cách ghép target và metrics như notebook train để so sánh.

Chọn GPU trong runtime, upload TSV vào session, rồi sửa DATA và OUTPUT bên dưới. Không mount Drive; không cần GPU local. Colab clone main vào `/content/semeval`; Kaggle dùng `/kaggle/working/semeval` (bật GPU và Internet).

## 1. Lấy code và cài thư viện
Chạy phần này trước khi import torch/transformers. Nếu session đã import các thư viện đó trước khi cài, khởi động lại session rồi chạy từ đầu.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = "https://github.com/iamtro-2006/semeval.git"
BRANCH = "main"
BASE = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
REPO_DIR = BASE / "semeval"
BASE.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO, str(REPO_DIR)], check=True)
else:
    current = subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip()
    changes = subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain", "--untracked-files=no"], text=True).strip()
    if current != BRANCH or changes:
        raise RuntimeError("Existing clone has a different branch or local edits. Choose a new REPO_DIR.")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

WORK = REPO_DIR / "task 8" / "openjev"
assert (WORK / "run.py").is_file(), WORK
print("Code:", WORK)
subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], check=True)


In [ ]:
import importlib.metadata
from packaging.version import Version

packages = ["transformers==5.17.0", "accelerate==1.13.0",
            "safetensors>=0.7,<1", "tqdm>=4.67,<5"]
try:
    torch_version = Version(importlib.metadata.version("torch").split("+")[0])
except importlib.metadata.PackageNotFoundError:
    torch_version = Version("0")
# Preserve a recent Colab/Kaggle PyTorch stack. Upgrade the matched trio only if needed.
if torch_version < Version("2.8"):
    packages += ["torch==2.8.0", "torchvision==0.23.0", "torchaudio==2.8.0"]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)

import importlib.util
spec = importlib.util.spec_from_file_location("openjev_run", WORK / "run.py")
run = importlib.util.module_from_spec(spec)
spec.loader.exec_module(run)
print("Ready. Model weights have not been downloaded yet.")


## 2. Đường dẫn và cấu hình
DATA là TSV muốn dự đoán: có các cột StereoQueerEval_id, yt_title, yt_description, yt_comment. Cột target là tùy chọn: có nhãn thì tính metrics, thiếu nhãn thì chỉ lưu dự đoán.

Mặc định `THRESHOLD = 0.5`, `TUNE_ON_DEV = False`: ngưỡng cố định trước khi chạy, không dùng nhãn để chọn ngưỡng. Nếu muốn so sánh với bản fine-tune có chọn ngưỡng, bật TUNE_ON_DEV và nhập DEV riêng có nhãn; DEV không được trùng ID/comment với DATA. Khi đó kết quả được đánh dấu `zero_shot_dev_threshold`, không gọi là zero-shot dùng ngưỡng cố định. Không dùng test để chọn ngưỡng.

Mỗi lần so sánh dùng một OUTPUT khác để không ghi đè kết quả.


In [ ]:
DATA = str(BASE / "dev.tsv")  # Colab: /content/dev.tsv; Kaggle: /kaggle/input/.../dev.tsv
OUTPUT = str(BASE / "zero-shot")
MODEL = "openjev/OpenJev-Flash-9B"
MAX_LENGTH = 2048
THRESHOLD = 0.5

TUNE_ON_DEV = False
DEV = ""  # Only set this when tuning; DATA should then be a separate held-out set.

job = run.ZeroShot(data=DATA, output=OUTPUT, dev=DEV if TUNE_ON_DEV else "",
                   model=MODEL, max_length=MAX_LENGTH)


## 3. Kiểm tra input
Kiểm tra GPU BF16, tokenizer và độ dài từng prompt trước khi tải weights. Không tự cắt title, description hoặc comment. Nếu vượt MAX_LENGTH, tăng giới hạn theo thông báo rồi tạo job và chạy check lại.


In [ ]:
job.check()


## 4. Chạy zero-shot
Mỗi comment có 9 câu yes/no cho identity và 1 câu chọn scope. Đọc xác suất từ logits A/B tại vị trí trả lời đầu tiên, không sinh reasoning hoặc JSON. Không identity nào vượt ngưỡng thì ghi abstain và giữ mẫu trong đánh giá.

Đây là restricted A/B softmax, chưa calibration cho dataset này và không phải các con số benchmark của API OpenJev gốc. Để so sánh hiệu quả fine-tune, dùng cùng DATA, model revision, câu hỏi và chính sách ngưỡng. Bản fine-tune mặc định chọn ngưỡng trên dev. Để so sánh cùng protocol, dùng DATA=test và bật TUNE_ON_DEV với cùng dev cho zero-shot. Ngưỡng cố định 0.5 là baseline không tune bằng nhãn.


In [ ]:
metrics = job.infer(threshold=THRESHOLD, tune_on_dev=TUNE_ON_DEV)
print("Results:", OUTPUT)


## 5. Tải kết quả
Chỉ lưu settings.json (checkpoint revision, câu hỏi và protocol), predictions.csv (ID, gold nếu có, nhãn, xác suất) và metrics.json. Không lưu nội dung comment, bản sao dataset, adapter hoặc checkpoint training. Kết quả trên session có thể mất khi phiên kết thúc; tải ZIP về máy để giữ lại.


In [ ]:
import zipfile

folder = Path(OUTPUT).resolve()
archive = folder.parent / (folder.name + ".zip")
if not (folder / "metrics.json").is_file():
    raise FileNotFoundError("Run inference successfully before exporting.")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for name in ("settings.json", "predictions.csv", "metrics.json"):
        z.write(folder / name, arcname=name)
print("ZIP:", archive)

try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(str(archive)))
else:
    files.download(str(archive))
